# 测试 Yolo C2f

In [1]:
import sys
sys.path.append("..")
import set_env

/media/pc/data/lxw/ai/tvm-book


In [2]:
from pathlib import Path
import torch
from torch.onnx import utils
import numpy as np
from d2py.utils.file import mkdir
from models.c2f import C2f

temp_dir = Path(".temp")
temp_dir.mkdir(exist_ok=True)

In [3]:
model = C2f(3, 64, shortcut=True)
model.eval()

shape = 1, 3, 48, 80
input_name = "x"
dtype = "float32"
data_np = np.random.rand(*shape).astype(dtype)
output_name = "test"
xx = torch.rand(*shape, dtype=torch.float32, requires_grad=False)
utils.export(
    model,               # torch 模型
    xx,                         # 模型输入或者对于多个输入，使用元组
    temp_dir/f"./{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
    export_params=True,        # 将训练后的参数权重存储在模型文件内
    opset_version=17,          # 导出模型的 ONNX 版本
    do_constant_folding=True,  # 是否执行常量折叠以进行优化
    input_names = [input_name],    # 模型的输入名称
    output_names = ['output'], # 模型的输出名称
    keep_initializers_as_inputs=True,
    # export_modules_as_functions=True,
    verbose=True,
    # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
    #               'output' : {0 : 'batch_size'}}
)
data = torch.rand(shape).float()
trace = torch.jit.trace(model.eval(), data).eval()
torch.jit.save(trace, temp_dir/f"{output_name}.pt")

Exported graph: graph(%x : Float(1, 3, 48, 80, strides=[11520, 3840, 80, 1], requires_grad=0, device=cpu),
      %onnx::Conv_47 : Float(64, 3, 1, 1, strides=[3, 1, 1, 1], requires_grad=0, device=cpu),
      %onnx::Conv_48 : Float(64, strides=[1], requires_grad=0, device=cpu),
      %onnx::Conv_50 : Float(32, 32, 3, 3, strides=[288, 9, 3, 1], requires_grad=0, device=cpu),
      %onnx::Conv_51 : Float(32, strides=[1], requires_grad=0, device=cpu),
      %onnx::Conv_53 : Float(32, 32, 3, 3, strides=[288, 9, 3, 1], requires_grad=0, device=cpu),
      %onnx::Conv_54 : Float(32, strides=[1], requires_grad=0, device=cpu),
      %onnx::Conv_56 : Float(64, 96, 1, 1, strides=[96, 1, 1, 1], requires_grad=0, device=cpu),
      %onnx::Conv_57 : Float(64, strides=[1], requires_grad=0, device=cpu)):
  %/cv1/conv/Conv_output_0 : Float(1, 64, 48, 80, strides=[245760, 3840, 80, 1], requires_grad=1, device=cpu) = onnx::Conv[dilations=[1, 1], group=1, kernel_shape=[1, 1], pads=[0, 0, 0, 0], strides=[1, 1]

In [4]:
import tvm
from tvm import relay
model = torch.jit.load(temp_dir/f"{output_name}.pt")
mod, params = relay.frontend.from_pytorch(model, [(input_name, shape)])

In [5]:
with tvm.transform.PassContext(opt_level=3, required_pass=["InferType"]):
    run_mod = relay.quantize.prerequisite_optimize(mod, params)

In [6]:
import tvm
from tvm import relay
from tvm.relay.testing import run_opt_pass
from tvm.relay.dataflow_pattern import (
    wildcard, is_op, 
    is_tuple_get_item,
    is_constant, 
    is_tuple,
    DFPatternCallback,
    rewrite
)
from tvm.relay import op as _op
from tvm.relay import transform as _transform
from tvm.relay.frontend.common import infer_value

In [12]:
class Split2StridedSliceRewrite(DFPatternCallback):
    def __init__(self):
        super().__init__()
        self.x = wildcard()
        self.split = is_op("split")(self.x) #.has_attr({"axis": 1})
        # self.tuple_get_item0 = is_tuple_get_item(self.split, 0)
        # self.tuple_get_item1 = is_tuple_get_item(self.split, 1)
        # self.output = is_tuple([self.tuple_get_item0, self.tuple_get_item1])
        self.pattern = self.split #self.output
        self.ops = []

    def callback(self, pre, post, node_map):
        x = node_map[self.x][0]
        split = node_map[self.split][0]
        # indices_or_sections = split.attrs['indices_or_sections']
        # axis = split.attrs['axis']
        shape = _transform.InferTypeLocal(x).shape
        # {split.body.tuple_value} => {split.body.size}
        print(f"XX: {x} => {split}")
        self.ops.append(split)
        # if len(indices_or_sections) == 1:
        #     begin = [0] * len(shape)
        #     begin[axis] = int(indices_or_sections[0])
        #     ret = relay.strided_slice(x, begin=begin, end=shape)
        #     _transform.InferTypeLocal(ret)
        #     return ret
        return post

In [15]:
transform = Split2StridedSliceRewrite()
expr = run_mod["main"]
expr = rewrite(transform, expr)

XX: free_var %x: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=aten::_convolution_0.x:0:0 */;
%0 = nn.conv2d(%x, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 1, 1), float32] */, padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 48, 80), float32] */;
%1 = add(%0, meta[relay.Constant][1] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 48, 80), float32] */;
%2 = sigmoid(%1) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */;
multiply(%1, %2) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */
 => free_var %x: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=aten::_convolution_0.x:0:0 */;
%0 = nn.conv2d(%x, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 1, 1), float32] */, padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 48, 80), float32] */;
%1 = add(%0, meta[relay.Constant][1] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 48, 80), floa

In [26]:
sp = transform.ops[0]
_mod = tvm.IRModule.from_expr(sp)

In [32]:
# print(relay.Tuple(sp))

In [33]:
print(_mod.script())

def @main(%x: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=aten::_convolution_0.x:0:0 */) {
  %0 = nn.conv2d(%x, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 1, 1), float32] */, padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 48, 80), float32] */;
  %1 = add(%0, meta[relay.Constant][1] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 48, 80), float32] */;
  %2 = sigmoid(%1) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */;
  %3 = multiply(%1, %2) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */;
  split(%3, indices_or_sections=[meta[runtime.BoxInt][0]], axis=1) /* ty=(Tensor[(1, 32, 48, 80), float32], Tensor[(1, 32, 48, 80), float32]) span=aten::split_with_sizes_0:0:0 */
}




In [24]:
_mod.show()

In [ ]:
@tvm.transform.module_pass(opt_level=1)
class SimplifyYoloC2F:
    """重写YoloC2F"""
    def __init__(self):
        self.ops = []
    def transform_module(self, mod, ctx):
        expr = rewrite(Split2StridedSliceRewrite(), mod["main"])
        # expr = rewrite(FuseSplitConcatRewrite(), mod["main"])
        # expr = rewrite(SplitItem2SliceConcatRewrite(), expr)
        return tvm.IRModule.from_expr(expr)

In [ ]:
# run_mod2 = SimplifyYoloC2F()(run_mod)

XX: free_var %x: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=aten::_convolution_0.x:0:0 */;
%0 = nn.conv2d(%x, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 1, 1), float32] */, padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 48, 80), float32] */;
%1 = add(%0, meta[relay.Constant][1] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 48, 80), float32] */;
%2 = sigmoid(%1) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */;
multiply(%1, %2) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */
 => free_var %x: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=aten::_convolution_0.x:0:0 */;
%0 = nn.conv2d(%x, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 1, 1), float32] */, padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 48, 80), float32] */;
%1 = add(%0, meta[relay.Constant][1] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 48, 80), floa

In [ ]:
ss

In [10]:
print(run_mod2["main"])

fn (%x: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=aten::_convolution_0.x:0:0 */) -> Tensor[(1, 64, 48, 80), float32] {
  %0 = nn.conv2d(%x, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 1, 1), float32] */, padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 48, 80), float32] */;
  %1 = add(%0, meta[relay.Constant][1] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 48, 80), float32] */;
  %2 = sigmoid(%1) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */;
  %3 = multiply(%1, %2) /* ty=Tensor[(1, 64, 48, 80), float32] span=aten::silu_0:0:0 */;
  %4 = split(%3, indices_or_sections=[meta[runtime.BoxInt][0]], axis=1) /* ty=(Tensor[(1, 32, 48, 80), float32], Tensor[(1, 32, 48, 80), float32]) span=aten::split_with_sizes_0:0:0 */;
  %5 = %4.1 /* ty=Tensor[(1, 32, 48, 80), float32] span=aten::split_with_sizes_0:0:0 */;
  %6 = nn.conv2d(%5, meta[relay.Constant][2] /* ty=Tensor[(32, 32, 3, 3), float32] */, padding=[1

In [ ]:
print(run_mod["main"])

SyntaxError: invalid syntax (4175519072.py, line 1)

In [ ]:
target = "llvm"
dev = tvm.device(target, 0)
exe = relay.create_executor(
    "graph", mod=run_mod, params=params, device=dev, target=target
).evaluate()
result = exe(**{input_name: data_np})

In [ ]:
target = "llvm"
dev = tvm.device(target, 0)
exe2 = relay.create_executor(
    "graph", mod=run_mod2, params=params, device=dev, target=target
).evaluate()
result2 = exe2(**{input_name: data_np})

In [ ]:
np.testing.assert_allclose(result.numpy(), result2.numpy())